# 01 — Data exploration

What the raw price series looks like before any of it becomes a picture. The goal here
is to know the shape of the data well enough that later results are interpretable: how
many bars there are, how the up/down split sits, and how much of the target is simply
the long upward drift of an index.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # import src/ from notebooks/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config
from src.data_pipeline.fetch_ohlc import fetch_ohlc

bars = fetch_ohlc(config.TICKER)
print(f'{len(bars)} bars  {bars.index[0].date()} -> {bars.index[-1].date()}')
bars.tail()

## Price history and the three chronological splits

Splits are by date, never random, and the shaded gaps are the embargo: windows there are
discarded because they would otherwise straddle a boundary and share candles across splits.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.semilogy(bars.index, bars['Close'], lw=0.8, color='#222')
for date, label, colour in [(config.TRAIN_END, 'train ends', '#2E7D32'),
                            (config.VAL_END, 'val ends', '#EF6C00')]:
    ax.axvline(pd.Timestamp(date), color=colour, ls='--', lw=1.4, label=label)
ax.set_title(f'{config.TICKER} close (log scale) and split boundaries')
ax.set_ylabel('close'); ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

## How much of the target is just drift?

The share of up-days is the accuracy a model gets by always predicting "up". Every
later accuracy number has to be read against this, not against 50%.

In [ ]:
from src.downstream_signal.labels import base_rate, next_day_direction

y = next_day_direction(bars)
print(f'up-days overall: {base_rate(y):.4f}')
for name, mask in [('train', y.index <= config.TRAIN_END),
                   ('val', (y.index > config.TRAIN_END) & (y.index <= config.VAL_END)),
                   ('test', y.index > config.VAL_END)]:
    print(f'  {name:<6} n={mask.sum():>5}  up-rate={y[mask].mean():.4f}')

## Daily return distribution

Fat tails and volatility clustering are why volatility features are in the baseline at all.

In [ ]:
rets = np.log(bars['Close']).diff().dropna()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(rets, bins=140, color='#3949AB', alpha=0.85)
axes[0].set_yscale('log'); axes[0].set_title('daily log returns (log-count)')
axes[1].plot(rets.index, rets.rolling(20).std() * np.sqrt(252), lw=0.7, color='#C62828')
axes[1].set_title('20-day realised volatility, annualised'); axes[1].grid(alpha=0.2)
plt.tight_layout(); plt.show()
print(rets.describe())

## One rendered chart

This is literally all the detector ever sees: 20 candles, no axes, no gridlines, no volume.

In [ ]:
from src.data_pipeline.render_charts import render_window

window = bars.iloc[-config.WINDOW:]
image, mapper = render_window(window)
plt.figure(figsize=(6, 6)); plt.imshow(image); plt.axis('off')
plt.title(f'{config.TICKER} to {window.index[-1].date()}'); plt.show()
print('image', image.shape)